# 01 · Tokenizer Lab — RUNG 4 (the tokenizer)

**Page 44** (`44-tokenization.html`) · LLM track · the input side of an LLM, de-mystified.

What this notebook does, top to bottom:
1. Loads the **real** Qwen3 tokenizer from Hugging Face (not the toy BPE demo on the page).
2. Reproduces the true **3-token** split of `strawberry` — and shows *why the model can't count the r's*.
3. Measures **ρ** (characters per token) on English, code, and **your own** jargon.
4. Prints **your corpus's total token count** — the number that feeds **page 49's** wall-clock estimate — and persists it to disk.

**SAFETY / footprint.** Read-only w.r.t. your system. **No GPU needed** — a tokenizer is pure CPU. The only thing written is a tiny JSON (`01_tokenizer_lab_output.json`) next to this notebook, holding your corpus token count for page 49. First run downloads the tokenizer + config only (a few MB — **no model weights**) into your HF cache; set `HF_HOME` to your NVMe first (setup page). ~2 minutes.

> **Rung 4 needs no Spark.** This runs on a laptop CPU. *On* the Spark, use your fresh **course** venv — never ComfyUI's (setup-page rule 1).

**Verified against** (constants §7 / brief-tooling-hardware, July 2026): `transformers 5.14.1`. The tokenizer API used here — `AutoTokenizer` / `AutoConfig` `.from_pretrained`, `.encode(add_special_tokens=False)`, `.decode` — is stable across the 5.x line.

In [ ]:
# --- version stamp (house style: a support request starts with real versions) ---
import sys, platform
print(f"python       {platform.python_version()}  ({sys.platform})")
try:
    import transformers
    print(f"transformers {transformers.__version__}")
except ImportError:
    transformers = None
    print("transformers NOT importable -- the model-loading cells will self-skip.")
    print("  install into your COURSE venv (NEVER ComfyUI's):")
    print("    uv pip install 'transformers==5.14.1'")
print("verified against: transformers 5.14.1  (constants section 7)")

# Tokenization is deterministic; we seed anyway so the whole track shares one habit.
import random
random.seed(42)

In [ ]:
# --- frozen numbers this notebook asserts against (constants.md, verbatim) ---
MODEL_ID = "Qwen/Qwen3-8B"
V_CONFIG = 151936          # Qwen3-8B config.json vocab_size   [VP, constants section 1.1]

# V is a byte-level BPE vocab PADDED for tensor-core alignment: 151936 = 128 x 1187.
assert V_CONFIG == 128 * 1187, "V must be a multiple of 128 (embedding padding)"
print(f"V (embedding rows, config.json) = {V_CONFIG:,} = 128 x 1187   [VP, constants section 1.1]")

## 1 · Load the real tokenizer — and the number that isn't what you think

The page freezes one assertion: **V = 151,936**. But there is an honest subtlety the course refuses to launder over. That 151,936 is the **model's** number — the `vocab_size` in `config.json`, i.e. the number of rows in the embedding matrix. The **tokenizer** can emit *fewer* distinct ids than that; the extra rows are padding so the `lm_head` matmul aligns to tensor-core tiles.

So we assert the **config** value (the tensor you actually pay memory for) and **report the tokenizer's real count beside it** — never assert one as if it were the other.

In [ ]:
if transformers is not None:
    from transformers import AutoTokenizer, AutoConfig

    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    cfg = AutoConfig.from_pretrained(MODEL_ID)

    # The assertion the spec freezes: the MODEL's vocab (config.json) is 151,936.
    assert cfg.vocab_size == V_CONFIG, f"config vocab {cfg.vocab_size} != {V_CONFIG}"
    print(f"config.vocab_size      = {cfg.vocab_size:,}   [VP, constants section 1.1]  -> asserted OK")

    # The honest gap: the tokenizer emits FEWER distinct ids than the embedding has rows.
    tok_len = len(tok)                 # includes added / special tokens
    print(f"len(tokenizer)         = {tok_len:,}   distinct ids the tokenizer can emit")
    print(f"embedding padding gap  = {V_CONFIG - tok_len:,} rows that exist for the hardware, not the language")
    print()
    print("Why they differ: the embedding is padded up to a multiple of 128 so the lm_head")
    print("matmul tiles cleanly on the tensor cores. Those extra rows are never emitted by")
    print("the tokenizer. The course asserts the CONFIG number (151,936 -- the tensor you pay")
    print("memory for) and prints the tokenizer's real count next to it, rather than passing")
    print("one off as the other.")
else:
    print("skipped: transformers not installed (see the stamp cell above).")

## 2 · `strawberry` → exactly 3 tokens

The page's predict box asks: 1, 3, or 10? Here is the real tokenizer's answer.

In [ ]:
if transformers is not None:
    word = "strawberry"
    ids = tok.encode(word, add_special_tokens=False)
    pieces = [tok.decode([i]) for i in ids]
    print(f"encode({word!r})  ->  {ids}")
    print(f"pieces            ->  {pieces}   ({len(ids)} tokens)")

    assert len(ids) == 3, f"expected the frozen 3-token split, got {len(ids)}: {pieces}"
    print()
    print("Exactly 3 tokens -- the page's predict answer, from the REAL tokenizer.")
    print("The model never sees 's-t-r-a-w-b-e-r-r-y'; it sees three fragment IDs. That is why")
    print("a model that writes flawless essays cannot reliably count the r's: the individual")
    print("letters were compressed away before layer 1. Counting letters is a property of a")
    print("representation the model does not have.")
else:
    print("skipped: transformers not installed.")

## 3 · ρ — characters per token

ρ = chars / token is the compression ratio. English prose sits near **ρ ≈ 4** — an **[EST]** rule of thumb (constants §9.5), *not* a measured law. Code and non-Latin scripts fall lower: the same information costs more tokens. Below we **measure** ρ on three strings — those numbers are `[MEA]`, the ~4 is the standing estimate.

In [ ]:
# rho = characters per token. A PURE function -- also exercised by the self-test cell.
def rho(n_chars, n_tokens):
    return n_chars / n_tokens if n_tokens else float("nan")

if transformers is not None:
    samples = {
        "English prose": "The quick brown fox jumps over the lazy dog near the riverbank at first light.",
        "Python code":   "def f(x):\n    return [i**2 for i in range(x) if i % 3 == 0]\n",
        "CLI / jargon":  "Set HF_HOME to the NVMe, then: uv pip install torch==2.13.0 --index-url ...",
    }
    print(f"{'kind':<16}{'chars':>7}{'tokens':>8}{'rho':>7}")
    print("-" * 38)
    for name, text in samples.items():
        n = len(tok.encode(text, add_special_tokens=False))
        print(f"{name:<16}{len(text):>7}{n:>8}{rho(len(text), n):>7.2f}")
    print()
    print("English lands near 4 [EST, constants section 9.5]; code and jargon fall lower.")
    print("The per-string numbers above are MEASURED [MEA]; the ~4 is the estimate, not a law.")
else:
    print("skipped: transformers not installed.")

## 4 · Your corpus → total token count (this feeds page 49)

Point `CORPUS_DIR` at a folder of your own text files. The notebook counts every token across all of them and **persists the total** to `01_tokenizer_lab_output.json`, which **page 49** reads to turn a training run into a wall-clock estimate.

In [ ]:
import os, json, glob

# Point this at a folder of your own text files. Leave as None to skip.
CORPUS_DIR = None            # e.g. r"C:\\Users\\you\\notes"  or  "/home/you/corpus"
EXTS = (".txt", ".md", ".py", ".rst", ".json", ".jsonl")

if transformers is not None and CORPUS_DIR:
    paths = [p for p in glob.glob(os.path.join(CORPUS_DIR, "**", "*"), recursive=True)
             if os.path.isfile(p) and p.lower().endswith(EXTS)]
    total_tokens = total_chars = 0
    for p in paths:
        try:
            text = open(p, encoding="utf-8", errors="ignore").read()
        except OSError:
            continue
        total_tokens += len(tok.encode(text, add_special_tokens=False))
        total_chars += len(text)

    r = rho(total_chars, total_tokens)
    print(f"files scanned      {len(paths):,}")
    print(f"total characters   {total_chars:,}")
    print(f"total tokens       {total_tokens:,}   <-- feeds page 49's wall-clock")
    print(f"corpus-wide rho    {r:.2f}  chars/token   [MEA on YOUR corpus]")

    out = {
        "model_id": MODEL_ID, "corpus_dir": CORPUS_DIR, "files": len(paths),
        "total_chars": total_chars, "total_tokens": total_tokens, "rho": r,
        "note": "consumed by page 49 (05_memory_ledger / wall-clock estimate)",
    }
    with open("01_tokenizer_lab_output.json", "w") as f:
        json.dump(out, f, indent=2)
    print("\npersisted -> 01_tokenizer_lab_output.json")
elif transformers is not None:
    print("CORPUS_DIR is None -- set it to a folder of your text files to measure your own")
    print("corpus and persist the token count that page 49 reads. (Skipped for now.)")
else:
    print("skipped: transformers not installed.")

## 5 · Why V costs you — the logits tensor

V is not free. It sets the width of every logits tensor and the `lm_head` matmul. Pure arithmetic, no model needed — and it explains a real API default you will meet in the training track.

In [ ]:
# GiB/GB discipline (constants section 0): a single tensor quoted alone is GB.
GB = 10**9
S = 2048                                   # one sequence
logits_bytes = S * V_CONFIG * 2            # bf16 = 2 bytes / element
assert logits_bytes == 622_329_856, logits_bytes

print(f"logits tensor, B=1 S={S}, bf16:")
print(f"  {S} x {V_CONFIG:,} x 2 = {logits_bytes:,} B = {logits_bytes/GB:.3f} GB   [DER, constants section 9.5]")
print("  ...for ONE tensor. fp32 cross-entropy doubles it to ~1.2 GB, and you need its")
print("  gradient too. That 622 MB is exactly why TRL 1.8 defaults loss_type='chunked_nll' --")
print("  it drops the -100 label positions BEFORE the lm_head matmul. A real, current default,")
print("  priced right here from V.")

## 6 · Self-test — no model, no network, no GPU

This cell is the path verified on the build machine (Windows, no torch/CUDA, no transformers). It exercises the pure arithmetic and the `rho()` function so a future edit can't silently drift a frozen number.

In [ ]:
# ============================================================================
# SELF-TEST -- runs with NO model, NO network, NO GPU. Pure arithmetic + rho().
# ============================================================================
def selftest():
    # 1. frozen config vocab and its 128-alignment
    assert V_CONFIG == 151936 == 128 * 1187
    # 2. rho() exact on the page's worked example: 2600 chars / 650 tokens == 4.0
    assert abs(rho(2600, 650) - 4.0) < 1e-12
    assert round(rho(2600, 650)) == 4          # the standing English estimate rounds to 4
    # 3. logits-tensor cost derivation (constants section 9.5)
    assert 2048 * V_CONFIG * 2 == 622_329_856
    # 4. the strawberry claim is a COUNT the live cell checks; freeze the expectation
    STRAWBERRY_TOKENS = 3
    assert STRAWBERRY_TOKENS == 3
    # 5. wall-clock feed sanity (illustrative; the real number is page 49's):
    #    total_tokens / throughput = seconds -> 6,969,590 tok at 6,969.59 tok/s = 1000 s
    assert abs(6_969_590 / 6969.59 - 1000.0) < 1e-6
    return "all self-test assertions passed"

print(selftest())
print(f"rho(2600 chars, 650 tokens)   = {rho(2600, 650):.4f} chars/token  [matches page worked example]")
print(f"V = {V_CONFIG:,} = 128 x 1187  (embedding padded for tensor cores)")
print(f"logits(B=1,S=2048,bf16)       = 622,329,856 B = 0.622 GB")
print(f"strawberry frozen token count = 3")